# ML Deployment — Assignment 3
### Building a Flask Prediction API

---
## Question 1
**Install Flask in a new Python virtual environment and create a file called `app.py` that displays 'Welcome to the Prediction API' at the root URL (/).**

### Step 1: Create Virtual Environment & Install Flask

Run the following commands in terminal:
```bash
# Create virtual environment
python -m venv venv

# Activate it (Windows)
venv\Scripts\activate

# Install Flask
pip install flask
```

### Step 2: Basic Flask App (Root URL)

The following code creates a basic Flask app with a welcome message at `/`:

In [ ]:
# --- Q1: Basic Flask App (assignment3/app.py) ---
# This is the starting point of our app.py

from flask import Flask

app = Flask(__name__)

@app.route('/')
def home():
    return 'Welcome to the Prediction API'

if __name__ == '__main__':
    app.run(debug=True)

# To run: python assignment3/app.py
# Visit: http://127.0.0.1:5000/
print("Q1 Code: Basic Flask app with root URL showing 'Welcome to the Prediction API'")

---
## Question 2
**Build a simple REST API in Flask called `/predict-price` that accepts a POST request with JSON input containing 'base_price' and 'discount' fields, and returns the final price after applying the discount.**

In [ ]:
# --- Q2: /predict-price endpoint ---
# Added to assignment3/app.py

from flask import Flask, request, jsonify

# @app.route('/predict-price', methods=['POST'])
def predict_price():
    data = request.get_json()

    # Validate input
    if not data or 'base_price' not in data or 'discount' not in data:
        return jsonify({
            'error': 'Please provide both "base_price" and "discount" in JSON body'
        }), 400

    base_price = data['base_price']
    discount = data['discount']

    # Calculate final price
    final_price = base_price - (base_price * discount / 100)

    return jsonify({
        'base_price': base_price,
        'discount_percent': discount,
        'final_price': round(final_price, 2)
    })

# Sample curl command to test:
# curl -X POST http://127.0.0.1:5000/predict-price -H "Content-Type: application/json" -d '{"base_price": 1000, "discount": 15}'

print("Q2 Code: /predict-price endpoint that calculates discounted price")
print("Example: base_price=1000, discount=15% -> final_price=850.0")

---
## Question 3
**Save a trained scikit-learn LinearRegression model (for example, predicting delivery time based on distance and order size) to a file using joblib, then load this model in your Flask app and use it to make predictions when the `/predict-delivery` endpoint receives a POST request with JSON input.**

### Step 1: Train and Save the Model

In [ ]:
import numpy as np
import joblib
import os
from sklearn.linear_model import LinearRegression

# Create assignment3 folder for all generated files
os.makedirs('assignment3', exist_ok=True)

# --- Training Data ---
# Features: [distance_km, order_size (num_items)]
# Target:   delivery_time (minutes)

X_train = np.array([
    [2,  1],    # 2 km, 1 item  -> 15 min
    [5,  2],    # 5 km, 2 items -> 25 min
    [1,  1],    # 1 km, 1 item  -> 10 min
    [8,  3],    # 8 km, 3 items -> 40 min
    [3,  2],    # 3 km, 2 items -> 20 min
    [10, 4],    # 10 km, 4 items -> 50 min
    [6,  1],    # 6 km, 1 item  -> 28 min
    [4,  3],    # 4 km, 3 items -> 30 min
    [7,  2],    # 7 km, 2 items -> 35 min
    [12, 5],    # 12 km, 5 items -> 60 min
])

y_train = np.array([15, 25, 10, 40, 20, 50, 28, 30, 35, 60])

# --- Train the model ---
model = LinearRegression()
model.fit(X_train, y_train)

print("Model trained successfully!")
print(f"R² Score: {model.score(X_train, y_train):.4f}")
print(f"Coefficients: distance={model.coef_[0]:.2f}, order_size={model.coef_[1]:.2f}")
print(f"Intercept: {model.intercept_:.2f}")

# --- Save the model ---
joblib.dump(model, 'assignment3/delivery_model.joblib')
print("\nModel saved as 'assignment3/delivery_model.joblib'")

# --- Quick test ---
test_input = np.array([[5, 2]])
prediction = model.predict(test_input)
print(f"\nTest: distance=5km, order_size=2 items -> Predicted delivery: {prediction[0]:.1f} min")

### Step 2: Load Model in Flask & Create /predict-delivery Endpoint

The model is loaded at app startup and used in the `/predict-delivery` route:

In [ ]:
# --- Q3: /predict-delivery endpoint (added to assignment3/app.py) ---

import joblib
import numpy as np

# Load model at startup
delivery_model = joblib.load('assignment3/delivery_model.joblib')

# @app.route('/predict-delivery', methods=['POST'])
def predict_delivery_basic():
    data = request.get_json()

    distance_km = data['distance_km']
    order_size = data['order_size']

    # Make prediction
    input_features = np.array([[distance_km, order_size]])
    predicted_time = delivery_model.predict(input_features)[0]

    return jsonify({
        'predicted_delivery_time_min': round(predicted_time, 1)
    })

# Test the loaded model
test = delivery_model.predict(np.array([[5, 2]]))
print(f"Model loaded and tested: 5km, 2 items -> {test[0]:.1f} min")

---
## Question 4
**Update your Flask API so that it returns a JSON response with both the predicted value and a custom message, similar to how Swiggy or Zomato shows estimated delivery time and a friendly note.**

In [ ]:
# --- Q4: Updated /predict-delivery with friendly message ---
# (This is the final version in assignment3/app.py)

# @app.route('/predict-delivery', methods=['POST'])
def predict_delivery_with_message():
    data = request.get_json()

    distance_km = data['distance_km']
    order_size = data['order_size']

    # Make prediction
    input_features = np.array([[distance_km, order_size]])
    predicted_time = round(delivery_model.predict(input_features)[0], 1)

    # Custom friendly message (like Swiggy/Zomato)
    if predicted_time <= 20:
        message = f"🚀 Lightning fast! Your order will arrive in ~{predicted_time} mins."
    elif predicted_time <= 35:
        message = f"🛵 On its way! Estimated delivery in ~{predicted_time} mins. Hang tight!"
    else:
        message = f"📦 Your order is being prepared. Expected delivery in ~{predicted_time} mins. Thanks for your patience!"

    return jsonify({
        'distance_km': distance_km,
        'order_size': order_size,
        'predicted_delivery_time_min': predicted_time,
        'message': message
    })

# Demo the friendly messages
print("Demo of friendly messages for different deliveries:\n")
for dist, items in [(2, 1), (5, 2), (10, 4)]:
    t = round(delivery_model.predict(np.array([[dist, items]]))[0], 1)
    if t <= 20:
        msg = f"🚀 Lightning fast! ~{t} mins"
    elif t <= 35:
        msg = f"🛵 On its way! ~{t} mins"
    else:
        msg = f"📦 Being prepared. ~{t} mins"
    print(f"  {dist}km, {items} items -> {t} min | {msg}")

---
## Question 5
**Run your Flask prediction API locally, then use Postman or curl to send a sample JSON request to your `/predict-delivery` endpoint and verify the response.**

*Hint: If you get a CORS or content-type error, check your request headers and Flask route methods.*

### Step 1: Start the Flask Server

Run this in terminal:
```bash
python assignment3/app.py
```

Output:
```
--- Flask Prediction API ---
Endpoints:
  GET  http://127.0.0.1:5000/
  POST http://127.0.0.1:5000/predict-price
  POST http://127.0.0.1:5000/predict-delivery
* Running on http://127.0.0.1:5000
```

### Step 2: Test with curl Commands

**Test 1: GET / (Welcome Message)**
```bash
curl http://127.0.0.1:5000/
```
**Response:**
```
Welcome to the Prediction API
```

---

**Test 2: POST /predict-price**
```bash
curl -X POST http://127.0.0.1:5000/predict-price -H "Content-Type: application/json" -d '{"base_price": 1000, "discount": 15}'
```
**Response:**
```json
{
  "base_price": 1000,
  "discount_percent": 15,
  "final_price": 850.0
}
```

---

**Test 3: POST /predict-delivery**
```bash
curl -X POST http://127.0.0.1:5000/predict-delivery -H "Content-Type: application/json" -d '{"distance_km": 5, "order_size": 2}'
```
**Response:**
```json
{
  "distance_km": 5,
  "message": "🛵 On its way! Estimated delivery in ~27.3 mins. Hang tight!",
  "order_size": 2,
  "predicted_delivery_time_min": 27.3
}
```

### Step 3: Test Programmatically with Python `requests` Library

⚠️ **Note:** Run this cell only while the Flask server is running in a separate terminal.

In [ ]:
import requests

BASE_URL = 'http://127.0.0.1:5000'

# --- Test 1: GET / ---
print("=" * 50)
print("TEST 1: GET /")
print("=" * 50)
response = requests.get(f'{BASE_URL}/')
print(f"Status: {response.status_code}")
print(f"Response: {response.text}")

# --- Test 2: POST /predict-price ---
print("\n" + "=" * 50)
print("TEST 2: POST /predict-price")
print("=" * 50)
response = requests.post(
    f'{BASE_URL}/predict-price',
    json={'base_price': 1000, 'discount': 15}
)
print(f"Status: {response.status_code}")
print(f"Response: {response.json()}")

# --- Test 3: POST /predict-delivery ---
print("\n" + "=" * 50)
print("TEST 3: POST /predict-delivery")
print("=" * 50)
response = requests.post(
    f'{BASE_URL}/predict-delivery',
    json={'distance_km': 5, 'order_size': 2}
)
print(f"Status: {response.status_code}")
result = response.json()
print(f"Response: {result}")
print(f"\n📍 Distance: {result['distance_km']} km")
print(f"📦 Order Size: {result['order_size']} items")
print(f"⏱️  Delivery Time: {result['predicted_delivery_time_min']} min")
print(f"💬 Message: {result['message']}")

---
## Files Structure

```
ML_DEPLOYMENT/
├── assignment3.ipynb                         <- This notebook (main file)
└── assignment3/                              <- All generated files
    ├── app.py                                <- Flask API (Q1 + Q2 + Q3 + Q4)
    ├── train_model.py                        <- Model training script (Q3)
    └── delivery_model.joblib                 <- Saved ML model (Q3)
```